In [4]:
import sys
sys.path.append('..')
from src.bq_client import run_query
import pandas as pd

- row counts for each table

In [3]:
df_counts = run_query("""
    SELECT 'users' AS table_name, COUNT(*) AS row_count 
    FROM `bigquery-public-data.thelook_ecommerce.users`
    UNION ALL
    SELECT 'orders', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.orders`
    UNION ALL
    SELECT 'order_items', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.order_items`
    UNION ALL
    SELECT 'products', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.products`
    UNION ALL
    SELECT 'events', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.events`
    UNION ALL
    SELECT 'inventory_items', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.inventory_items`
    UNION ALL
    SELECT 'distribution_centers', COUNT(*) 
    FROM `bigquery-public-data.thelook_ecommerce.distribution_centers`
""")
df_counts

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,row_count
0,orders,124690
1,products,29120
2,inventory_items,487585
3,users,100000
4,order_items,180520
5,events,2416574
6,distribution_centers,10


- understanding events table

In [4]:
df = run_query("""
    SELECT *
    FROM `bigquery-public-data.thelook_ecommerce.events`
    LIMIT 5
""")
print(df.to_string(index=False))
print("\nCOLUMNS:", list(df.columns))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


     id  user_id  sequence_number                           session_id                created_at      ip_address      city     state postal_code browser traffic_source     uri event_type
2394075     <NA>                3 6322d00d-98a7-411f-b0bc-61d2e5aba804 2020-11-15 06:14:00+00:00    188.90.5.218 São Paulo São Paulo   02220-000  Chrome        Adwords /cancel     cancel
1808851     <NA>                3 e11e4a16-8391-4da3-9b36-9f241a3c5a36 2023-03-17 01:15:00+00:00    73.79.126.98 São Paulo São Paulo   02675-031  Chrome          Email /cancel     cancel
2184038     <NA>                3 56863b99-abfe-4b94-ac7f-ad48f5dc21f7 2026-03-26 02:37:00+00:00  204.240.98.145 São Paulo São Paulo   02675-031  Chrome          Email /cancel     cancel
1625596     <NA>                3 1739888d-f30d-4e5c-a9c6-75a1d2338bcb 2023-12-01 11:54:00+00:00 147.156.226.136 São Paulo São Paulo   02675-031  Chrome        Adwords /cancel     cancel
1930285     <NA>                3 63b521c9-cad7-4698-a7a7-25f95a7

- event types and count

In [6]:
df = run_query("""
    SELECT event_type, COUNT(*) AS count
    FROM `bigquery-public-data.thelook_ecommerce.events`
    GROUP BY event_type
    ORDER BY count DESC
""")
print(df.to_string(index=False))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


event_type  count
   product 841328
department 591416
      cart 591124
  purchase 180520
    cancel 124866
      home  87320


- null rates in event table

In [7]:
df = run_query("""
    SELECT
        ROUND(COUNTIF(user_id IS NULL) / COUNT(*) * 100, 2) AS user_id_null_pct,
        ROUND(COUNTIF(session_id IS NULL) / COUNT(*) * 100, 2) AS session_id_null_pct,
        ROUND(COUNTIF(created_at IS NULL) / COUNT(*) * 100, 2) AS created_at_null_pct,
        ROUND(COUNTIF(event_type IS NULL) / COUNT(*) * 100, 2) AS event_type_null_pct,
        ROUND(COUNTIF(ip_address IS NULL) / COUNT(*) * 100, 2) AS ip_address_null_pct
    FROM `bigquery-public-data.thelook_ecommerce.events`
""")
print(df.to_string(index=False))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


 user_id_null_pct  session_id_null_pct  created_at_null_pct  event_type_null_pct  ip_address_null_pct
            46.54                  0.0                  0.0                  0.0                  0.0


- understanding orders table

In [42]:
df = run_query("""
    SELECT status, COUNT(*) AS count
    FROM `bigquery-public-data.thelook_ecommerce.orders`
    GROUP BY status
    ORDER BY count DESC
""")
print(df.to_string(index=False))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


    status  count
   Shipped  37532
  Complete  31041
Processing  25051
 Cancelled  18781
  Returned  12433


In [9]:
df = run_query("""
    SELECT
        MIN(created_at) AS earliest_order,
        MAX(created_at) AS latest_order
    FROM `bigquery-public-data.thelook_ecommerce.orders`
""")
print(df.to_string(index=False))

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


           earliest_order                     latest_order
2019-01-05 13:53:54+00:00 2026-06-24 00:48:06.823550+00:00


In [10]:
df = run_query("""
    SELECT 
        event_type,
        COUNT(*) AS total_events,
        COUNT(DISTINCT session_id) AS sessions_with_this_event,
        COUNT(DISTINCT user_id) AS unique_users
    FROM `bigquery-public-data.thelook_ecommerce.events`
    GROUP BY event_type
    ORDER BY total_events DESC
""")
df

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,event_type,total_events,sessions_with_this_event,unique_users
0,product,841328,680520,79987
1,department,591416,430608,79987
2,cart,591124,430316,79987
3,purchase,180520,180520,79987
4,cancel,124866,124866,0
5,home,87320,87320,62991


In [11]:
df = run_query("""
    SELECT
        CASE WHEN user_id IS NULL THEN 'anonymous' ELSE 'authenticated' END AS user_type,
        COUNT(*) AS event_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
    FROM `bigquery-public-data.thelook_ecommerce.events`
    GROUP BY user_type
""")
df

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_type,event_count,pct_of_total
0,authenticated,1291824,53.46
1,anonymous,1124750,46.54


In [30]:
# Find a real user with lots of events to use for manual verification tomorrow
df = run_query("""
    SELECT 
        user_id,
        COUNT(*) AS event_count,
        MIN(created_at) AS first_event,
        MAX(created_at) AS last_event
    FROM `bigquery-public-data.thelook_ecommerce.events`
    WHERE user_id IS NOT NULL
    GROUP BY user_id
    ORDER BY event_count DESC
    LIMIT 10
""")
df

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,user_id,event_count,first_event,last_event
0,24115,186,2025-12-09 04:50:26+00:00,2026-05-18 19:07:38+00:00
1,44516,170,2026-04-12 06:48:47+00:00,2026-05-29 13:41:48+00:00
2,32146,161,2024-06-22 08:08:29+00:00,2026-04-14 19:57:46+00:00
3,63435,156,2023-10-30 05:42:06+00:00,2025-03-03 04:38:43+00:00
4,38897,148,2020-05-14 17:07:30+00:00,2025-05-06 11:39:20+00:00
5,42306,148,2021-04-21 03:26:51+00:00,2026-01-24 07:11:35+00:00
6,20412,148,2024-12-14 02:40:41+00:00,2025-05-28 23:10:46+00:00
7,10459,139,2025-07-14 22:19:44+00:00,2026-05-13 20:45:01+00:00
8,62746,139,2021-03-30 03:09:07+00:00,2023-06-14 11:35:37+00:00
9,86071,139,2024-03-18 07:52:04+00:00,2026-01-31 00:20:30+00:00


- testing sessionization query

In [31]:
df_sessionized = run_query("""
    WITH event_gap AS (
        SELECT 
            user_id, session_id, event_type, created_at, sequence_number,
            TIMESTAMP_DIFF(
                created_at, 
                LAG(created_at) OVER (
                    PARTITION BY user_id 
                    ORDER BY created_at ASC
                ), 
                MINUTE
            ) AS mins_since_last_event
        FROM `bigquery-public-data.thelook_ecommerce.events`
        WHERE user_id IN (24115, 44516)
    ),
    session_boundary AS (
        SELECT *,
            CASE 
                WHEN mins_since_last_event IS NULL THEN 1 
                WHEN mins_since_last_event > 30    THEN 1
                ELSE 0 
            END AS is_new_session
        FROM event_gap
    ),
    assign_session_num AS (
        SELECT *,
            SUM(is_new_session) OVER (
                PARTITION BY user_id 
                ORDER BY created_at ASC
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW 
            ) AS session_sequence
        FROM session_boundary
    )
    SELECT 
        user_id, session_id, event_type, created_at, sequence_number,
        mins_since_last_event, is_new_session, session_sequence
    FROM assign_session_num
    ORDER BY user_id, created_at ASC
""")
df_sessionized.columns = df_sessionized.columns.str.lower()
print(f"Total rows: {len(df_sessionized)}")  

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Total rows: 356


- First event is always session 1

In [33]:
first_events = df_sessionized.groupby('user_id').first().reset_index()
assert (first_events['session_sequence'] == 1).all(), "FAIL: First event not session 1"
assert (first_events['is_new_session'] == 1).all(), "FAIL: First event not flagged as new session"
print("Check 1 passed: First events correctly flagged")

Check 1 passed: First events correctly flagged


- Every row with gap > 30 minutes has is_new_session = 1

In [34]:
boundary_rows = df_sessionized[df_sessionized['mins_since_last_event'] > 30]
assert (boundary_rows['is_new_session'] == 1).all(), "FAIL: Gap > 30 not flagged as new session"
print(f"Check 2 passed: {len(boundary_rows)} session boundaries correctly detected")

Check 2 passed: 41 session boundaries correctly detected


- Every row with gap ≤ 30 minutes has is_new_session = 0

In [35]:
within_session = df_sessionized[
    df_sessionized['mins_since_last_event'].notna() & 
    (df_sessionized['mins_since_last_event'] <= 30)
]
assert (within_session['is_new_session'] == 0).all(), "FAIL: Gap <= 30 incorrectly flagged"
print(f"Check 3 passed: {len(within_session)} within-session events correctly unflagged")

Check 3 passed: 313 within-session events correctly unflagged


- Session count is consistent with boundary count

In [37]:
for uid in [24115, 44516]:
    user_df = df_sessionized[df_sessionized['user_id'] == uid]
    n_sessions = user_df['session_sequence'].max()
    n_boundaries = user_df['is_new_session'].sum()
    assert n_sessions == n_boundaries, f"FAIL: user {uid} session count mismatch"
    print(f"user {uid}: {n_sessions} sessions, {len(user_df)} total events")

user 24115: 19 sessions, 186 total events
user 44516: 24 sessions, 170 total events


- Testing identity resolution query

In [23]:
df_mixed_sessions = run_query("""
    SELECT
        session_id,
        COUNTIF(user_id IS NULL) AS anonymous_events,
        COUNTIF(user_id IS NOT NULL) AS authenticated_events,
        MAX(user_id) AS authenticated_user_id
    FROM `bigquery-public-data.thelook_ecommerce.events`
    GROUP BY session_id
    HAVING anonymous_events > 0 AND authenticated_events > 0
    ORDER BY anonymous_events DESC
    LIMIT 10
""")
df_mixed_sessions

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,session_id,anonymous_events,authenticated_events,authenticated_user_id


In [24]:
df_check = run_query("""
    SELECT
        session_id,
        COUNT(DISTINCT user_id) AS distinct_user_ids,
        COUNTIF(user_id IS NULL) AS null_count,
        COUNTIF(user_id IS NOT NULL) AS not_null_count
    FROM `bigquery-public-data.thelook_ecommerce.events`
    GROUP BY session_id
    ORDER BY not_null_count DESC, null_count DESC
    LIMIT 20
""")
df_check

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,session_id,distinct_user_ids,null_count,not_null_count
0,12f1df3f-7408-4bf9-82b4-6084a40f05fc,1,0,13
1,d58e2e60-922a-4644-85f2-5606d35f7cf6,1,0,13
2,2f02a781-3277-473b-9019-87d2918d9782,1,0,13
3,3361cf63-7384-4d91-b15e-e8e8ceb0bc22,1,0,13
4,2a4980aa-b273-4123-a537-328bce776378,1,0,13
5,86bd9e8e-d53b-4d4e-9b9e-7d26c3d0fbd0,1,0,13
6,4e234273-d4f8-4f66-b6d2-f5ecb3ec499d,1,0,13
7,9878a0ca-9ce4-4cda-b72d-c3b6050c07d4,1,0,13
8,0e051e34-e067-419c-9462-bcc7eb498230,1,0,13
9,7208151d-d184-47b2-bf55-8f285e9418ff,1,0,13


- Testing the funnel query

In [38]:
df_funnel = run_query("""
    WITH session_flags AS (
        SELECT
            session_id,
            MIN(created_at) AS session_start,
            COUNT(*) AS total_events,
            MAX(CASE WHEN event_type = 'home'       THEN 1 ELSE 0 END) AS reached_home,
            MAX(CASE WHEN event_type = 'department' THEN 1 ELSE 0 END) AS reached_department,
            MAX(CASE WHEN event_type = 'product'    THEN 1 ELSE 0 END) AS reached_product,
            MAX(CASE WHEN event_type = 'cart'       THEN 1 ELSE 0 END) AS reached_cart,
            MAX(CASE WHEN event_type = 'purchase'   THEN 1 ELSE 0 END) AS reached_purchase
        FROM `bigquery-public-data.thelook_ecommerce.events`
        GROUP BY session_id
    )
    SELECT 
        *,
        CASE 
            WHEN reached_purchase   = 1 THEN 'Completed Purchase'
            WHEN reached_cart       = 1 THEN 'Abandoned at Checkout'
            WHEN reached_product    = 1 THEN 'Abandoned at Cart'
            WHEN reached_department = 1 THEN 'Abandoned at Product'
            WHEN reached_home       = 1 THEN 'Abandoned at Department'
            ELSE 'Bounced Immediately'
        END AS abandonment_stage
    FROM session_flags
""")

df_funnel.columns = df_funnel.columns.str.lower()
total_sessions = len(df_funnel)
print(f"Total rows (sessions): {total_sessions:,}")

# checking uniqueness 
assert df_funnel['session_id'].is_unique, "FAIL: session_id is not unique per row!"
print("Check 1 passed: One row per session verified.")

# ensuring that every user landed somewhere valid
no_milestones = df_funnel[
    (df_funnel[['reached_home', 'reached_department', 'reached_product', 'reached_cart', 'reached_purchase']].sum(axis=1) == 0)
]
assert len(no_milestones) == 0, f"FAIL: Found {len(no_milestones)} sessions with no stages hit."
print("Check 2 passed: Every session hit at least one milestone.")

# calculating traffic per page
print("\nSESSION TRAFFIC PER PAGE:")
print("-" * 50)
for stage in ['reached_home', 'reached_department', 'reached_product', 'reached_cart', 'reached_purchase']:
    count = df_funnel[stage].sum()
    print(f"  {stage:<20} : {count:>9,} sessions ({count/total_sessions*100:>5.1f}%)")

print("\nBREAKDOWN OF WHERE USERS QUIT:")
print("-" * 50)
print(df_funnel['abandonment_stage'].value_counts(normalize=True) * 100)

c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Total rows (sessions): 682,025
Check 1 passed: One row per session verified.
Check 2 passed: Every session hit at least one milestone.

SESSION TRAFFIC PER PAGE:
--------------------------------------------------
  reached_home         :    88,179 sessions ( 12.9%)
  reached_department   :   431,551 sessions ( 63.3%)
  reached_product      :   682,025 sessions (100.0%)
  reached_cart         :   432,205 sessions ( 63.4%)
  reached_purchase     :   182,025 sessions ( 26.7%)

BREAKDOWN OF WHERE USERS QUIT:
--------------------------------------------------
abandonment_stage
Abandoned at Checkout    36.681940
Abandoned at Cart        36.629156
Completed Purchase       26.688904
Name: proportion, dtype: float64


c:\Users\Admin\Desktop\Summer\project\thelook-ecommerce-analytics\venv\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Total sessions: 682,025

📊 ABANDONMENT STAGE BREAKDOWN:
abandonment_stage
Abandoned at Cart       250180
Abandoned at Product    249820
Completed Purchase      182025
Name: count, dtype: int64

abandonment_stage
Abandoned at Cart       36.7
Abandoned at Product    36.6
Completed Purchase      26.7
Name: proportion, dtype: float64
